# Module 02 — Vision (Image Input)

Send images to **`claude-opus-4-8`** as **base64 image blocks** through the Agent Platform and have it describe and analyze them.

This builds on **Modules 00–01** and uses the same **`AnthropicVertex` / ADC** path — so these calls are also captured by the BigQuery logging from Module 00.

## Part B — Bootstrap

Setup mirrors Module 00 — see that module for full detail (prerequisites, ADC, logging).

In [ ]:
%pip install --quiet "anthropic[vertex]" pillow matplotlib

In [ ]:
# ===== CONDENSED BOOTSTRAP (mirrors Module 00) =====
import subprocess
from anthropic import AnthropicVertex

def _default_project() -> str:
    try:
        out = subprocess.run(
            ["gcloud", "config", "get-value", "project"],
            capture_output=True, text=True, timeout=15,
        ).stdout.strip()
        if out and out != "(unset)":
            return out
    except Exception:
        pass
    return "<YOUR_PROJECT_ID>"

PROJECT_ID = _default_project()   # or hard-code: PROJECT_ID = "my-project-id"
LOCATION   = "global"
MODEL      = "claude-opus-4-8"

assert PROJECT_ID and PROJECT_ID != "<YOUR_PROJECT_ID>", (
    "PROJECT_ID is still the placeholder. Run `gcloud config set project <id>` "
    "so it auto-detects, or edit PROJECT_ID directly."
)

client = AnthropicVertex(project_id=PROJECT_ID, region=LOCATION)
print(f"✅ Client ready (project={PROJECT_ID}, region={LOCATION}, model={MODEL}).")

## Part C — Create a self-contained sample image

To keep the notebook self-contained, we **generate** a small sample image locally (no external URLs, no copyrighted assets) and save it under the existing `assets/` folder. Claude will analyze this file in the next steps.

In [ ]:
import os
import matplotlib
matplotlib.use("Agg")  # non-interactive backend; just write a file
import matplotlib.pyplot as plt

os.makedirs("assets", exist_ok=True)
IMAGE_PATH = "assets/sample_chart.png"

months = ["Jan", "Feb", "Mar", "Apr", "May"]
values = [120, 90, 160, 140, 210]

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(months, values, color="#4C72B0")
ax.set_title("Monthly Active Users (thousands)")
ax.set_xlabel("Month")
ax.set_ylabel("Users (thousands)")
ax.bar_label(bars)  # label each bar with its value
fig.tight_layout()
fig.savefig(IMAGE_PATH, dpi=100)
plt.close(fig)

print(f"✅ Saved sample image to: {os.path.abspath(IMAGE_PATH)}")

## Part D — Send the image (base64)

For image input, a message's `content` is a **list** that mixes an **image block** and a **text block**. The image block carries a **base64** `source` with a declared `media_type` (here `image/png`); the text block is your prompt about the image.

In [ ]:
import base64

def image_to_base64(path: str) -> str:
    """Read a local image file and return its base64-encoded data."""
    with open(path, "rb") as f:
        return base64.standard_b64encode(f.read()).decode()

img_b64 = image_to_base64(IMAGE_PATH)

message = client.messages.create(
    model=MODEL,
    max_tokens=512,
    messages=[{
        "role": "user",
        "content": [
            {
                "type": "image",
                "source": {
                    "type": "base64",
                    "media_type": "image/png",
                    "data": img_b64,
                },
            },
            {"type": "text", "text": "Describe this chart in 2-3 sentences."},
        ],
    }],
)
print(message.content[0].text)

## Part E — Practical analysis + notes

Vision is **analytical**, not just descriptive — you can ask targeted questions and pull out structured facts (values, comparisons, trends) rather than a generic caption.

In [ ]:
message = client.messages.create(
    model=MODEL,
    max_tokens=256,
    messages=[{
        "role": "user",
        "content": [
            {
                "type": "image",
                "source": {
                    "type": "base64",
                    "media_type": "image/png",
                    "data": img_b64,
                },
            },
            {"type": "text", "text": "Which month has the highest value, and what is it? Answer in one line."},
        ],
    }],
)
print(message.content[0].text)

### Formats & limits

- **Supported formats:** JPEG, PNG, GIF, and WebP (set `media_type` to match).
- **Large images are downsampled** by the API, so very high resolution buys little — and costs more tokens.
- **Base64 is the universal path** on the Agent Platform; you can also send multiple image blocks in one message.
- See the Anthropic **Vision** docs (https://docs.claude.com/en/docs/build-with-claude/vision) and confirm current size/count limits there, as they may change.

## Part F — Recap

- Image input uses a **content list** mixing an **image block** (base64 `source` + `media_type`) and a **text block**.
- We generated a **self-contained** sample chart locally into `assets/` — no external assets needed.
- Vision is **analytical**: ask targeted questions to extract specific facts, not just descriptions.
- Formats: JPEG/PNG/GIF/WebP; large images are downsampled; base64 is the universal path — confirm limits in the Vision docs.

**Next: Module 03 — PDF / document input.**